### 1.1 Подключение

In [ ]:
# как в дз
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Моменты времени считаем в UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

### 2. Анализ доступных данных в БД для выбора гипотезы

In [ ]:
bounds = q("""
    select 'регистрации (clients.registered_at)'      as event, min(registered_at) as first_at, max(registered_at) as last_at, count(*) as n
    from prime.clients
    union all
    select 'подписки (subscriptions.started_at)',     min(started_at), max(started_at), count(*)
    from prime.subscriptions
    union all
    select 'платежи (payments.paid_at, все статусы)', min(paid_at), max(paid_at), count(*)
    from prime.payments
""")
bounds

In [ ]:
clients = q("""
    select client_id, registered_at, acquisition_channel
    from prime.clients
    order by client_id
""")

subs = q("""
    select subscription_id, owner_client_id, tariff_id, started_at, ended_at, is_trial, cancel_reason
    from prime.subscriptions
    order by subscription_id
""")

print(f"clients: {len(clients):>8} строк, регистрации {clients['registered_at'].min()} … {clients['registered_at'].max()}")
print(f"subs:    {len(subs):>8} строк, старты {subs['started_at'].min()} … {subs['started_at'].max()}")
print(f"         последняя отмена {subs['ended_at'].max()}")
print("подписок без владельца в clients:", (~subs["owner_client_id"].isin(clients["client_id"])).sum())

In [ ]:
last_day = max(clients["registered_at"].max(), subs["started_at"].max(), subs["ended_at"].max())
first_month = clients["registered_at"].min().normalize().replace(day=1)
last_month = last_day.normalize().replace(day=1)
months = pd.date_range(first_month, last_month, freq="MS")

# Первая подписка каждого клиента — для «новых подписчиков».
first_sub = subs.groupby("owner_client_id")["started_at"].min()

rows = []
for start in months:
    end = start + pd.offsets.MonthBegin(1)          # правый конец не включён

    registered_total = int((clients["registered_at"] < end).sum())
    new_clients = int(((clients["registered_at"] >= start) & (clients["registered_at"] < end)).sum())

    # Подписка активна в месяце: началась до его конца и не закончилась до его начала.
    alive = subs[(subs["started_at"] < end) & (subs["ended_at"].isna() | (subs["ended_at"] >= start))]
    active_clients = int(alive["owner_client_id"].nunique())

    new_subscribers = int(((first_sub >= start) & (first_sub < end)).sum())

    rows.append({
        "month": start.strftime("%Y-%m"),
        "registered_total": registered_total,
        "new_clients": new_clients,
        "active_clients": active_clients,
        "new_subscribers": new_subscribers,
        "complete": end <= last_day.normalize() + pd.Timedelta(days=1),   # False — данные месяца неполные
    })

monthly = pd.DataFrame(rows).set_index("month")
monthly["active_share_%"] = (monthly["active_clients"] / monthly["registered_total"] * 100).round(2)
monthly["new_clients_share_%"] = (monthly["new_clients"] / monthly["registered_total"] * 100).round(2)
monthly["new_among_active_%"] = (monthly["new_subscribers"] / monthly["active_clients"] * 100).round(2)

# Контроль: новые клиенты по месяцам в сумме дают всех клиентов.
assert monthly["new_clients"].sum() == len(clients), "новые клиенты не сходятся с числом клиентов"
assert monthly["registered_total"].iloc[-1] == len(clients), "база последнего месяца не равна числу клиентов"

monthly

In [ ]:
rows = []
for start in months:
    end = start + pd.offsets.MonthBegin(1)          # правый конец не включён

    # Активна на начало месяца: началась раньше начала и не закончилась до него
    at_start = subs[(subs["started_at"] < start) & (subs["ended_at"].isna() | (subs["ended_at"] >= start))]
    cancels_from_start = int((at_start["ended_at"] < end).sum())     # NaT сравнивается как False

    started = (subs["started_at"] >= start) & (subs["started_at"] < end)
    ended = (subs["ended_at"] >= start) & (subs["ended_at"] < end)
    at_end = (subs["started_at"] < end) & (subs["ended_at"].isna() | (subs["ended_at"] >= end))

    rows.append({
        "month": start.strftime("%Y-%m"),
        "active_start": len(at_start),
        "new_subs": int(started.sum()),
        "cancels": int(ended.sum()),
        "cancels_from_start": cancels_from_start,
        "active_end": int(at_end.sum()),
        "complete": end <= last_day.normalize() + pd.Timedelta(days=1),
    })

churn = pd.DataFrame(rows).set_index("month")
churn["churn_%"] = (churn["cancels_from_start"] / churn["active_start"] * 100).round(2)

# Баланс: активные на конец = на начало + новые - отмены. Должен сходиться в каждом месяце.
balance = churn["active_start"] + churn["new_subs"] - churn["cancels"] - churn["active_end"]
assert (balance == 0).all(), f"баланс не сходится в месяцах: {list(balance[balance != 0].index)}"

print("активных на конец истории:", subs["ended_at"].isna().sum(), "(в описании базы: 167 978)")
churn

## Выбор участка для анализа

Я посмотрела динамику прироста новых клиентов по месяцам (январь 2024 – август 2026): долю активных и новых клиентов, а затем баланс подписок «активные на начало + новые − отмены = активные на конец»

Приток стабилен: около 10–11 тысяч новых подписок в месяц. Отток из активных на начало месяца держится около 4 % (и медленно снижается. Отмены растут органично вместе с тем, что ежемесячно наращивается клиентская база.
- В августе 2026 отток 17,46 % против 3,92 % в июле. Отменились 33 576 подписок из активных на начало месяца, при обычном оттоке ожидали бы около 7,5 тысячи, то есть лишних отмен около 26 тысяч. Активная база впервые упала: с 192 341 до 167 978 (−24 363). До этого она росла каждый месяц.
Поэтому для своего анализа я решила взять участок за июль-август 2026 г. и проанализировать отток (отмены) подписок



## **Бизнес-вопрос ** 
# **Почему в агусте 2026  упала выручка от подписок и что с этим делать?
Зерно: одна строка = одна подписка; канал берём у владельца, клиентов без канала держим отдельной строкой «без канала».
**Как будем раскладывать.**
1. Где: отмены по каналам, партнёрам, тарифам и причинам (сумма частей равна итогу).
2. За счёт чего: отмены = активные на начало × доля отменивших; изменение делим на эффект базы, структуры и ставки оттока.
3. Почему: причины отмены (`cancel_reason`) и день, когда произошёл всплеск.

**Ограничения данных.**
- Платежи есть только с 6 января 2025, а подписки с января 2024, поэтому отток считаем по `subscriptions`, а платежи используем только для денежной оценки.
- Август неполный (данные до 30 августа), поэтому сравниваем осторожно.
- Причина отмены это категория из базы, а не доказанная мотивация клиента: расчёт показывает, где вырос отток, но не доказывает, почему клиенты ушли.